# Iterators and Generators in JavaScript

## Definition

Iterators and generators are mechanisms that bring the concept of iteration directly into the core language, allowing you to customize how data sequences are traversed and how function execution is controlled.

An **iterator** is an object designed to pull data from a container one piece at a time, whereas a **generator** is a special type of function that simplifies creating iterators by allowing you to pause and resume code execution.

## 1. Iterators and the Iteration Protocol

An object becomes an **iterator** when it implements the **Iterator Protocol**. This means the object must have a `next()` method that returns an object with two properties:

- `value` — The next value in the sequence.
- `done` — A boolean that is `true` if the iteration is complete, and `false` otherwise.

An object becomes **iterable** if it implements the `[Symbol.iterator]` method, which returns an iterator. Built-in iterables include `Array`, `String`, `Map`, `Set`, and `NodeList`.

### Creating a Custom Iterator

This example creates a custom iterable range object without using generators:

```javascript
const customRange = {
  start: 1,
  end: 3,
  [Symbol.iterator]() {
    let current = this.start;
    let last = this.end;

    // Returns the iterator object
    return {
      next() {
        if (current <= last) {
          return { value: current++, done: false };
        }
        return { value: undefined, done: true };
      }
    };
  }
};

// Consuming via for...of loop
for (const num of customRange) {
  console.log(num); // Outputs: 1, then 2, then 3
}
```

## 2. Generators: The Elegant Alternative

Manually maintaining internal state in custom iterators can be error-prone. Generator functions provide a powerful, cleaner syntax to achieve the exact same behavior.

- **Syntax:** Defined using the `function*` syntax.
- **Behavior:** When called, a generator function does not run its code immediately. Instead, it returns a special Generator object that acts as both an iterable and an iterator.
- **The `yield` Keyword:** Code executes until it hits a `yield` statement, where it pauses execution, preserves its context, and outputs the value. Execution resumes only when `next()` is called again.

### Re-writing the Range Example with a Generator

```javascript
function* rangeGenerator(start, end) {
  for (let i = start; i <= end; i++) {
    yield i; // Pauses execution and returns the value
  }
}

const iterator = rangeGenerator(1, 3);

console.log(iterator.next()); // { value: 1, done: false }
console.log(iterator.next()); // { value: 2, done: false }
console.log(iterator.next()); // { value: 3, done: false }
console.log(iterator.next()); // { value: undefined, done: true }
```

Notice how much state-tracking boilerplate disappears compared to the manual `customRange` example above — the `current` variable and the `done` check are handled implicitly by the generator's paused execution context.

## 3. Key Conceptual Differences

| Feature | Iterators | Generators |
|---|---|---|
| Definition | An object implementing a `.next()` method. | A function (`function*`) that yields values. |
| State Tracking | Must be manually managed via external variables. | Automatically managed by the JavaScript runtime. |
| Execution | Continuous method calls. | Can pause and resume at explicit intervals. |
| Syntactic Helpers | Relies entirely on custom object structures. | Uses native `yield` and `yield*` expressions. |

## 4. Advanced Generator Features

### Infinite Sequences (Lazy Evaluation)

Because generators compute their values on demand (lazily), they can model infinite loops without freezing the system or overloading memory.

```javascript
function* generateIds() {
  let id = 1;
  while (true) {
    yield id++;
  }
}

const idStream = generateIds();
console.log(idStream.next().value); // 1
console.log(idStream.next().value); // 2
```

> **Caution:** Never spread (`[...idStream]`) or `for...of` over an infinite generator without a `break` condition — since it never reaches `done: true`, this will hang or crash the process. Infinite generators should always be consumed with manual `.next()` calls, or paired with `.take(n)` (see Iterator Helpers below).

### Two-Way Communication

The `.next()` method accepts an optional argument. Passing a value into `.next(value)` injects that value back into the generator function as the result of the currently evaluated `yield` expression.

```javascript
function* interactiveGenerator() {
  const reply = yield "Do you want to continue?";
  yield `You said: ${reply}`;
}

const game = interactiveGenerator();
console.log(game.next().value);      // "Do you want to continue?"
console.log(game.next("Yes").value); // "You said: Yes"
```

### Delegation with `yield*`

You can delegate execution from one generator function to another iterable object (like an array or another generator) using `yield*`.

```javascript
function* combinedGenerator() {
  yield* [1, 2];                // Delegates to an Array iterator
  yield* rangeGenerator(3, 4);  // Delegates to another Generator
}
console.log([...combinedGenerator()]); // [1, 2, 3, 4]
```

### Early Termination: `.return()` and `.throw()`

Beyond `.next()`, generator objects expose two more control methods worth knowing:

- **`.return(value)`** — Forces the generator to finish immediately, as if a `return` statement were hit at the current `yield`. Useful for cleanup (e.g., breaking out of a `for...of` loop early calls this automatically).
- **`.throw(error)`** — Injects an exception at the current paused `yield` point, which can be caught with a `try...catch` inside the generator body.

```javascript
function* demo() {
  try {
    yield 1;
    yield 2;
  } finally {
    console.log("Cleanup ran");
  }
}

const gen = demo();
console.log(gen.next());     // { value: 1, done: false }
console.log(gen.return(99)); // "Cleanup ran" logs, then { value: 99, done: true }
```

## 5. Built-In Iterator Helpers

Modern JavaScript runtimes support **Iterator Helpers**. These allow you to chain methods directly onto iterators and generators — much like standard arrays — without pulling the entire sequence into memory at once:

- `.map(fn)` / `.filter(fn)`
- `.take(n)` — grabs the first `n` items
- `.drop(n)` — skips the first `n` items

```javascript
function* naturals() {
  let n = 1;
  while (true) yield n++;
}

// Lazily take the first 5 even numbers, without ever
// materializing the infinite sequence in memory
const result = naturals()
  .filter(n => n % 2 === 0)
  .take(5)
  .toArray();

console.log(result); // [2, 4, 6, 8, 10]
```

This is the key practical payoff of the lazy-evaluation model — array-like ergonomics over sequences that could be infinite or arbitrarily large, without ever building the full array in memory.

## A Note on Async Generators

Not in the original material, but worth flagging since it's a natural next step: JavaScript also supports **async generators**, defined with `async function*` and consumed with `for await...of`. They combine the pause/resume model of generators with awaited asynchronous values — commonly used for streaming data, such as reading paginated API results or file streams chunk by chunk.

```javascript
async function* fetchPages(url) {
  let page = 1;
  while (true) {
    const res = await fetch(`${url}?page=${page++}`);
    const data = await res.json();
    if (data.items.length === 0) return;
    yield data.items;
  }
}
```

## Quick Recap

| Concept | One-line summary |
|---|---|
| Iterator Protocol | An object with `.next()` returning `{ value, done }` |
| Iterable | An object with `[Symbol.iterator]()` returning an iterator |
| Generator function | `function*` — pauses/resumes via `yield`, auto-manages state |
| `yield` | Pauses execution and emits a value |
| `yield*` | Delegates iteration to another iterable/generator |
| `.next(value)` | Resumes execution, injecting `value` as the yield's result |
| `.return()` / `.throw()` | Force-finish or inject an error into a paused generator |
| Iterator Helpers | `.map()`, `.filter()`, `.take()`, `.drop()` — lazy, array-like chaining |
| Async generators | `async function*` + `for await...of` — for streaming async data |